In [ ]:
import numpy
import pandas

import uproot
import awkward

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

from showerreco.geometry import ShowerElementHolder, SpacePoint, point
from showerreco.tools import start_position, pca_direction, shower_length
from showerreco.utils import add_3D_shower_cone

##### Import

In [ ]:
CAF_FILE = "/Users/triozzi/Analysis/numine/showerreco/data/Scrubbed_20260329T162934_20260330T054850-G4Step2Var_20260415T141743-NullVar_20260415T153925-MCstage0Var_20260415T160149-MCstage1Var.flat.caf.root"

BRANCHES = [
    # slice identity
    "rec.slc.self",
    "rec.slc.vertex.x",
    "rec.slc.vertex.y",
    "rec.slc.vertex.z",
    # spacepoint 3-D position and PFP assignment
    "rec.slc.reco.hit.spacepoint.XYZ.x",
    "rec.slc.reco.hit.spacepoint.XYZ.y",
    "rec.slc.reco.hit.spacepoint.XYZ.z",
    "rec.slc.reco.hit.spacepoint.pfpID",
    # hit-level quantities
    "rec.slc.reco.hit.cryoID",
    "rec.slc.reco.hit.tpcID",
    "rec.slc.reco.hit.planeID",
    "rec.slc.reco.hit.wireID",
    "rec.slc.reco.hit.peakAmplitude",
    "rec.slc.reco.hit.peakTime",
    # PFP stuff
    "rec.slc.reco.pfp.id",
    "rec.slc.reco.pfp.slcID"
]

with uproot.open(CAF_FILE) as f:
    tree = f["recTree"]
    data = tree.arrays(BRANCHES, library="awkward")

##### Raw SpacePoints

In [ ]:
EVENT_IDX = 2

X = numpy.array(data['rec.slc.reco.hit.spacepoint.XYZ.x'][EVENT_IDX])
Y = numpy.array(data['rec.slc.reco.hit.spacepoint.XYZ.y'][EVENT_IDX])
Z = numpy.array(data['rec.slc.reco.hit.spacepoint.XYZ.z'][EVENT_IDX])
ID = numpy.array(data['rec.slc.reco.hit.spacepoint.pfpID'][EVENT_IDX])
cID = numpy.array(data['rec.slc.reco.hit.cryoID'][EVENT_IDX])
pID = numpy.array(data['rec.slc.reco.hit.planeID'][EVENT_IDX])
t = numpy.array(data['rec.slc.reco.hit.peakTime'][EVENT_IDX])
Q = numpy.array(data['rec.slc.reco.hit.peakAmplitude'][EVENT_IDX])

vtxx = numpy.array(data['rec.slc.vertex.x'][EVENT_IDX])
vtxy = numpy.array(data['rec.slc.vertex.y'][EVENT_IDX])
vtxz = numpy.array(data['rec.slc.vertex.z'][EVENT_IDX])

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 3.5), ncols=2, layout='constrained')

ids = numpy.unique(ID)
cmap = matplotlib.colormaps['tab20']  # or 'tab10', etc.
norm = matplotlib.colors.BoundaryNorm(
    boundaries=numpy.arange(ids.min(), ids.max() + 1., 1),
    ncolors=cmap.N
)

# XY
ax = axes[0]
sc0 = ax.scatter(X, Y, marker='.', c=ID, cmap=cmap, norm=norm)
ax.set(
  xlabel = 'x [cm]',
  ylabel = 'y [cm]',
)
ax.scatter(vtxx, vtxy, marker='x', color='red')

# ZY
ax = axes[1]
ax.scatter(Z, Y, marker='.', c=ID, cmap=cmap, norm=norm)
ax.set(
  xlabel = 'z [cm]',
)
ax.scatter(vtxz, vtxy, marker='x', color='red')

# gfx
cbar = fig.colorbar(
    sc0,
    ax=axes,
    location='right',
    shrink=1.0,
    pad=0.02
)


plt.show()

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 3.5), ncols=2, layout='constrained')

# mask for the PFPs and cryo you want
VTX_ID = 0
MASK_NUE = ((ID == 0) | (ID == 1)) & (cID == 0)
X_pfp = X[MASK_NUE]
Y_pfp = Y[MASK_NUE]
Z_pfp = Z[MASK_NUE]
ID_pfp = ID[MASK_NUE]

# XY
ax = axes[0]
sc0 = ax.scatter(X_pfp, Y_pfp, marker='.', s=5, c=ID_pfp, cmap='jet')
ax.set(
  xlabel = 'x [cm]',
  ylabel = 'y [cm]',
)
ax.scatter(vtxx[VTX_ID], vtxy[VTX_ID], marker='o', facecolors='none', edgecolors='black', s=50)

# ZY
ax = axes[1]
ax.scatter(Z_pfp, Y_pfp, marker='.', s=5, c=ID_pfp, cmap='jet')
ax.set(
  xlabel = 'z [cm]',
)
ax.scatter(vtxz[VTX_ID], vtxy[VTX_ID], marker='o', facecolors='none', edgecolors='black', s=50)

plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import numpy as np

# mask for the PFPs and cryo you want
VTX_ID = 0
MASK_NUE = ((ID == 0) | (ID == 1)) & (cID == 0)
X_pfp = X[MASK_NUE]
Y_pfp = Y[MASK_NUE]
Z_pfp = Z[MASK_NUE]
ID_pfp = ID[MASK_NUE]

fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect((1.25, 1, 0.85))
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95)

sc = ax.scatter(
    X_pfp, Z_pfp, Y_pfp,
    c           = ID_pfp,
    s           = 4,
    marker      = '.',
    depthshade  = False,
    cmap        = 'jet',
    label       = 'PFPs'
)

ax.scatter(
    vtxx[VTX_ID], vtxz[VTX_ID], vtxy[VTX_ID],
    marker='o', facecolors='none', edgecolors='black', s=50,
    label = 'vertex'
)

# gfx
ax.set(
    facecolor = (0, 0, 0, 0),   # transparent axes background
    xlabel = '$X$ [cm]',
    ylabel = '$Z$ [cm]',
    zlabel = '$Y$ [cm]',
)
fig.patch.set_alpha(0.0)    # transparent figure background
ax.grid(False)  # clean look: remove grid
for axis in (ax.xaxis, ax.yaxis, ax.zaxis): # remove pane fills (the “3D box walls”)
    axis.pane.set_facecolor((1, 1, 1, 0))
    axis.pane.set_edgecolor((1, 1, 1, 0))

# ticks
ax.tick_params(axis='both', which='major', labelsize=10, pad=2)
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(2))
ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))
ax.zaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))

# grid
ax.grid(True)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis._axinfo["grid"]["linewidth"] = 0.25
    axis._axinfo["grid"]["linestyle"] = "-"
    axis._axinfo["grid"]["color"] = (0.7, 0.7, 0.7, 0.4)

ax.legend()

# move views...
ax.view_init(elev=20, azim=10)
ax.set_proj_type('ortho')

plt.show()
fig.savefig("plots/ShowerCharacterization_SpacePoints.pdf", dpi = 300)
fig.savefig("plots/ShowerCharacterization_SpacePoints.png", dpi = 300)

##### Running shower reconstruction tools...

In [ ]:
# reconstructed vertex
VTX_ID = 0
VERTEX = point(vtxx[VTX_ID], vtxy[VTX_ID], vtxz[VTX_ID])

# build space points
MASK_ELECTRON = (ID == 0) & (cID == 0)
X_pfp = X[MASK_ELECTRON]
Y_pfp = Y[MASK_ELECTRON]
Z_pfp = Z[MASK_ELECTRON]
Q_pfp = Q[MASK_ELECTRON]
t_pfp = t[MASK_ELECTRON]
N_sp  = len(X_pfp)

SPACEPOINTS = [
  SpacePoint(
    position = point(X_pfp[i], Y_pfp[i], Z_pfp[i]),
    charge = Q_pfp[i],
    time = t_pfp[i]
  )
  for i in range(N_sp)  
]

In [ ]:
HOLDER = ShowerElementHolder()

In [ ]:
# first tool: start position from vertex
rc1 = start_position.run(
    vertex      = VERTEX,
    spacepoints = SPACEPOINTS,
    holder      = HOLDER,
    verbose     = True,
)
print('Tool succeeded?', rc1)

# get shower start position
START = HOLDER.get_element('ShowerStartPosition')

In [ ]:
# second tool: PCA-based direction
# this is charge-weighted and oriented from the start position
rc2 = pca_direction.run(
    spacepoints          = SPACEPOINTS,
    holder               = HOLDER,
    charge_weighted      = True,
    use_start_position   = True,
    electron_lifetime_ms = 3.0,
    sampling_rate_us     = 0.5,
    verbose              = True,
)
print('Tool succeeded?', rc2)

PCA = HOLDER.get_element("ShowerPCA")
DIRECTION = HOLDER.get_element("ShowerDirection")  # unit vector
EIGENVALUES = PCA.eigenvalues

print(f"Reconstructed direction: {DIRECTION}")
print(f"PCA eigenvalues: {EIGENVALUES}")

# normalized direction
u = numpy.array(DIRECTION) 
u = u / numpy.linalg.norm(u)

In [ ]:
# third tool: length, end, opening angle
rc3 = shower_length.run(
    spacepoints = SPACEPOINTS,
    holder      = HOLDER,
    percentile  = 0.95,
    verbose     = True,
)

LENGTH = HOLDER.get_element("ShowerLength")
ANGLE  = HOLDER.get_element("ShowerOpeningAngle")

In [ ]:
# mask for the PFPs and cryo you want
VTX_ID = 0
MASK_NUE = ((ID == 0) | (ID == 1)) & (cID == 0)
X_pfp = X[MASK_NUE]
Y_pfp = Y[MASK_NUE]
Z_pfp = Z[MASK_NUE]
ID_pfp = ID[MASK_NUE]

fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect((1.25, 1, 0.85))
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95)

sc = ax.scatter(
    X_pfp, Z_pfp, Y_pfp,
    c           = ID_pfp,
    s           = 4,
    marker      = '.',
    depthshade  = False,
    cmap        = 'jet',
    label       = 'PFPs'
)

ax.scatter(
    vtxx[VTX_ID], vtxz[VTX_ID], vtxy[VTX_ID],
    marker='o', facecolors='none', edgecolors='black', s=50,
    label = 'vertex'
)

# reconstructed start
ax.scatter(
    START[0], START[2], START[1],
    marker  = '^', 
    s       = 30,
    color   = 'red',
    label   = 'shower start'
)

# reconstructed PCA-based direction
ax.plot(
    [START[0], START[0] + LENGTH*u[0]],
    [START[2], START[2] + LENGTH*u[2]],
    [START[1], START[1] + LENGTH*u[1]],
    color       = 'red',
    linewidth   = 2,
    label       = 'PCA axis'
)

ax.scatter(
    START[0] + LENGTH*u[0], START[2] + LENGTH*u[2], START[1] + LENGTH*u[1],
    marker  = 'v', 
    s       = 30,
    color   = 'red',
    label   = 'shower end'
)

proxy = add_3D_shower_cone(ax, START, u, LENGTH, ANGLE, ax_order=(0, 2, 1))

# gfx
ax.set(
    facecolor = (0, 0, 0, 0),   # transparent axes background
    xlabel = '$X$ [cm]',
    ylabel = '$Z$ [cm]',
    zlabel = '$Y$ [cm]',
)
fig.patch.set_alpha(0.0)    # transparent figure background
ax.grid(False)  # clean look: remove grid
for axis in (ax.xaxis, ax.yaxis, ax.zaxis): # remove pane fills (the “3D box walls”)
    axis.pane.set_facecolor((1, 1, 1, 0))
    axis.pane.set_edgecolor((1, 1, 1, 0))

# ticks
ax.tick_params(axis='both', which='major', labelsize=10, pad=2)
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(2))
ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))
ax.zaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))

# grid
ax.grid(True)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis._axinfo["grid"]["linewidth"] = 0.25
    axis._axinfo["grid"]["linestyle"] = "-"
    axis._axinfo["grid"]["color"] = (0.7, 0.7, 0.7, 0.4)

ax.legend()

# move views...
ax.view_init(elev=20, azim=10)
ax.set_proj_type('ortho')

plt.show()
fig.savefig("plots/ShowerCharacterization_SpacePoints_PCA.pdf", dpi = 300)
fig.savefig("plots/ShowerCharacterization_SpacePoints_PCA.png", dpi = 300)

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 3.5), ncols=2, layout='constrained')

# mask for the PFPs and cryo you want
VTX_ID = 0
MASK_NUE = ((ID == 0) | (ID == 1)) & (cID == 0)
X_pfp = X[MASK_NUE]
Y_pfp = Y[MASK_NUE]
Z_pfp = Z[MASK_NUE]
ID_pfp = ID[MASK_NUE]

# XY
ax = axes[0]
sc0 = ax.scatter(X_pfp, Y_pfp, marker='.', s=5, c=ID_pfp, cmap='jet')
ax.set(
  xlabel = 'x [cm]',
  ylabel = 'y [cm]',
)
ax.scatter(vtxx[VTX_ID], vtxy[VTX_ID], marker='o', facecolors='none', edgecolors='black', s=50, label='vertex')
ax.scatter(
    START[0], START[1],
    marker  = 'x', 
    s       = 50,
    color   = 'black',
    label   = 'shower start'
)
ax.plot(
    [START[0], START[0] + L*u[0]],
    [START[1], START[1] + L*u[1]],
    c = 'red', lw = 2, label = 'PCA axis', zorder = -3,
)

# ZY
ax = axes[1]
ax.scatter(Z_pfp, Y_pfp, marker='.', s=5, c=ID_pfp, cmap='jet')
ax.set(
  xlabel = 'z [cm]',
)
ax.scatter(vtxz[VTX_ID], vtxy[VTX_ID], marker='o', facecolors='none', edgecolors='black', s=50, label='vertex')
ax.scatter(
    START[2], START[1],
    marker  = 'x', 
    s       = 50,
    color   = 'black',
    label   = 'shower start'
)
ax.plot(
    [START[2], START[2] + L*u[2]],
    [START[1], START[1] + L*u[1]],
    c = 'red', lw = 2, label = 'PCA axis', zorder = -3,
)

ax.legend()

plt.show()